In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install bitsandbytes

In [ ]:
!pip install -q --upgrade transformers torch accelerate bitsandbytes sentence-transformers wikipedia-api wikipedia faiss-cpu requests tavily-python

In [ ]:
!pip install --upgrade torch torchvision --quiet
!pip install sentence-transformers --quiet

In [ ]:
!pip install -U transformers peft sentence-transformers

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --force-reinstall -q

In [2]:
# @title Swahili-Gemma RAG System - V5 (corrected & complete)
# Main fixes addressing V4 test failures:
# - repetition_penalty 1.3 → 1.18 (prevents number explosion)
# - temperature 0.15 when context is used → better adherence to facts
# - much stronger system prompt (exact numbers, no questions, no opinions unless explicit)
# - improved extract_model_response (catches number spam better, fallback message if empty)
# - removed "unadhani" from creative list → treated as opinion → usually uses context
# - sentence cap 5 instead of 4 + better joining logic
# - minor robustness improvements
!pip install -q transformers torch accelerate wikipedia-api wikipedia sentence-transformers faiss-cpu requests tavily-python
import torch
import wikipediaapi
import wikipedia
import numpy as np
import requests
import os
import re
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Tuple, Optional
from kaggle_secrets import UserSecretsClient
os.environ["TAVILY_API_KEY"] = UserSecretsClient().get_secret("TAVILY_API_KEY")
# =============================
# DEVICE & MODEL LOADING
# =============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {device}")
print("\n📥 Loading fine-tuned Swahili-Gemma model...")
start = time.time()
MODEL_PATH = "/kaggle/input/notebooks/briangreenheart/finetuninggood/swahili-gemma-finetuned/merged_model"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    local_files_only=True,
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"✅ Model loaded in {time.time() - start:.2f} seconds")
print("\n📥 Loading embedding model...")
torch.cuda.empty_cache()
embedding_model = SentenceTransformer('intfloat/multilingual-e5-large-instruct')
print("✅ Embedding model loaded")
# =============================
# GENERATION SETTINGS (improved defaults)
# =============================
generation_config = {
    "max_new_tokens": 220,
    "do_sample": True,
    "temperature": 0.30, # default / no-context path
    "top_p": 0.92,
    "top_k": 50,
    "repetition_penalty": 1.18, # lowered from 1.3 — critical fix
    "pad_token_id": tokenizer.eos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "use_cache": True,
}
print("\n⚙️ Generation Settings:")
for k, v in generation_config.items():
    if k not in ['pad_token_id', 'eos_token_id', 'use_cache']:
        print(f" • {k}: {v}")
# =============================
# CREATIVE / GRAMMAR DETECTION
# =============================
CREATIVE_PREFIXES = [
    "andika", "tunga", "badilisha", "fasiri", "tafsiri",
    "sahihisha", "unda", "ongeza", "punguza", "fupisha", "panua",
    "sema kwa", "translate", "eleza maana", "fafanua"
    # deliberately removed: "unadhani", "unafikiri" → now treated more carefully
]
def is_creative_or_grammar_query(query: str) -> bool:
    q = query.lower().strip()
    for prefix in CREATIVE_PREFIXES:
        if prefix in q:
            print(f" 🎨 Creative/grammar detected ('{prefix}') — skipping RAG")
            return True
    return False
# =============================
# WIKIPEDIA & ENTITY MAPPINGS (kept from original)
# =============================
class WikipediaKnowledgeBase:
    def __init__(self, user_agent='SwahiliRAG/1.0'):
        self.wiki_en = wikipediaapi.Wikipedia(language='en', user_agent=user_agent)
        self.wiki_sw = wikipediaapi.Wikipedia(language='sw', user_agent=user_agent)
        self.documents = []
        self.index = None
        self.page_cache = {}
        self.entity_mappings = {
            "gavana wa mombasa": "Abdulswamad Nassir",
            "mombasa governor": "Abdulswamad Nassir",
            "barack obama": "Barack Obama",
            "nairobi": "Nairobi",
            "raila odinga": "Raila Odinga",
            "hassan joho": "Hassan Joho",
            "william ruto": "William Ruto",
            "rais wa kenya": "William Ruto",
            "president of kenya": "William Ruto",
            "kenya": "Kenya",
            "tanzania": "Tanzania",
            "dar es salaam": "Dar es Salaam"
        }
    def get_best_search_term(self, query: str) -> str:
        query_lower = query.lower()
        for key, value in self.entity_mappings.items():
            if key in query_lower:
                print(f" 🔍 Mapped → '{value}'")
                return value
        words = [w for w in query_lower.strip('?').split() if w not in {"wa", "ni", "nani", "wapi", "gani"}]
        return " ".join(words[:3]) or query
    def search_and_fetch(self, query: str, max_results: int = 3) -> List[Dict]:
        if query in self.page_cache:
            return self.page_cache[query]
        articles = []
        term = self.get_best_search_term(query)
        print(f" 🔎 Searching: '{term}'")
        try:
            wikipedia.set_lang("en")
            for title in wikipedia.search(term, results=max_results):
                page = self.wiki_en.page(title)
                if page.exists():
                    articles.append({'title': page.title, 'content': page.summary, 'url': page.fullurl, 'language': 'en'})
                    break
            wikipedia.set_lang("sw")
            for title in wikipedia.search(term, results=2):
                page = self.wiki_sw.page(title)
                if page.exists():
                    articles.append({'title': page.title, 'content': page.summary, 'url': page.fullurl, 'language': 'sw'})
                    break
        except Exception as e:
            print(f" ⚠️ Wikipedia error: {e}")
        self.page_cache[query] = articles
        return articles
    def create_vector_store(self, articles: List[Dict]) -> int:
        self.documents = []
        for article in articles:
            chunks = [article['content'][i:i+500] for i in range(0, len(article['content']), 450)]
            for i, chunk in enumerate(chunks):
                self.documents.append({
                    'title': article['title'],
                    'content': chunk,
                    'url': article['url'],
                    'language': article.get('language', 'en'),
                    'chunk_id': i
                })
        if self.documents:
            texts = [f"passage: {d['content']}" for d in self.documents]
            embeddings = embedding_model.encode(texts, show_progress_bar=False)
            dim = embeddings.shape[1]
            self.index = faiss.IndexFlatL2(dim)
            self.index.add(embeddings.astype('float32'))
        return len(self.documents)
    def retrieve_relevant_context(self, query: str, k: int = 3) -> List[Dict]:
        if not self.index or not self.documents:
            return []
        q_emb = embedding_model.encode([f"query: {query}"])
        _, indices = self.index.search(q_emb.astype('float32'), k)
        return [self.documents[i] for i in indices[0] if i < len(self.documents)]
# =============================
# TAVILY + DDG (simplified)
# =============================
class TavilyRetriever:
    def __init__(self):
        self.api_key = os.environ.get("TAVILY_API_KEY", "")
        self.endpoint = "https://api.tavily.com/search"
    def search(self, query: str, max_results: int = 3) -> Optional[Dict]:
        if not self.api_key:
            print(" ⚠️ No TAVILY_API_KEY")
            return None
        try:
            r = requests.post(self.endpoint, json={
                "api_key": self.api_key,
                "query": query,
                "search_depth": "basic",
                "max_results": max_results,
                "include_answer": True
            }, timeout=12)
            data = r.json()
            if data.get("answer"):
                return {"content": data["answer"], "source": "Tavily", "url": ""}
            results = data.get("results", [])
            if results:
                txt = "\n\n".join(f"[{r.get('title','')}] {r.get('content','')}" for r in results)
                return {"content": txt, "source": "Tavily", "url": results[0].get("url","")}
        except Exception as e:
            print(f" Tavily error: {e}")
        return None
class DuckDuckGoRetriever:
    def search(self, query: str) -> Optional[Dict]:
        try:
            r = requests.get("https://api.duckduckgo.com/", params={
                "q": query, "format": "json", "no_html": 1
            }, timeout=6)
            data = r.json()
            if data.get("AbstractText"):
                return {"content": data["AbstractText"], "source": "DDG", "url": data.get("AbstractURL","")}
            if data.get("Answer"):
                return {"content": data["Answer"], "source": "DDG", "url": ""}
        except:
            pass
        return None
# =============================
# POST-PROCESSING (improved)
# =============================
def extract_model_response(full_response: str) -> str:
    if "\nmodel\n" in full_response:
        resp = full_response.split("\nmodel\n")[-1].strip()
    else:
        resp = full_response.strip()
    # Remove junk
    resp = re.sub(r'\[[\w\s\/\-]+\]', '', resp)
    resp = re.sub(r'^(Answer:|Jibu:|Jibu sahihi:)\s*', '', resp, flags=re.I)
    # Better sentence splitting
    sentences = re.split(r'(?<=[.!?])\s+', resp)
    clean = []
    seen = set()
    for s in sentences:
        s = s.strip()
        if len(s) < 4: continue
        norm = re.sub(r'\s+', ' ', s.lower().strip('.,!?'))
        if norm in seen: continue
        seen.add(norm)
        clean.append(s)
    clean = clean[:5]
    final = ' '.join(s.strip() for s in clean if s.strip()).strip()
    # Catch obvious garbage
    if re.search(r'(\d+\.){10,}', final) or len(final.split()) > 120 and '.' in final * 10:
        return "Samahani, nimepata hitilafu ya uundaji. Tafadhali jaribu tena."
    return final if final else "Samahani, sikupata jibu sahihi. Unaweza kuuliza tena?"
# =============================
# CONFIDENCE PROBE (kept similar)
# =============================
def should_use_rag(query: str, confidence_threshold: float = 0.68) -> Tuple[bool, str, float]:
    q_lower = query.lower()
    factual_kw = ["nani", "wapi", "lini", "idadi", "mwaka", "tarehe", "sasa", "karne", "202", "milioni"]
    is_factual = any(kw in q_lower for kw in factual_kw)
    is_short = len(query.split()) <= 5
    if is_short and not is_factual:
        print(" 💬 Short casual query — skipping RAG")
        return False, "", 1.0
    formatted = f"<start_of_turn>user\n{query}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = extract_model_response(txt)
    has_placeholder = any(x in answer.lower() for x in ['???', 'unknown', '[jina', '[tarehe'])
    too_short = len(answer.split()) < 4
    top_conf = 0.95 # dummy — real probe is noisy anyway
    use_rag = is_factual or has_placeholder or too_short
    print(f" 📊 Probe confidence rough: {top_conf:.2f} | placeholder: {has_placeholder} | short: {too_short}")
    return use_rag, answer, top_conf
# Generic page & event helpers (kept)
GENERIC_WIKI_TITLES = {"kenya", "nairobi", "mombasa", "kisumu", "nakuru",
    "tanzania", "dar es salaam", "dodoma", "arusha", "mwanza", "zanzibar",
    "uganda", "kampala",
    "rwanda", "kigali",
    "burundi", "gitega", "bujumbura",
    "ethiopia", "addis ababa",
    "somalia", "mogadishu",
    "djibouti", "eritrea", "asmara",
    "east africa", "africa", "horn of africa", "great lakes", "swahili", "kiswahili",
    "african union", "eac", "east african community"}
EVENT_KEYWORDS = ["2020","2021","2022","2023","2024","2025","2026","mkutano","uchaguzi","karne","mwaka","tarehe","leo","sasa","hivi karibuni","idadi","takwimu"]
def is_event_or_specific_query(q: str) -> bool:
    return any(kw in q.lower() for kw in EVENT_KEYWORDS)
def top_doc_is_generic(docs: List[Dict]) -> bool:
    if not docs: return False
    return docs[0].get('title','').lower().strip() in GENERIC_WIKI_TITLES
# =============================
# MAIN FUNCTION
# =============================
def generate_with_rag(
    query: str,
    kb: WikipediaKnowledgeBase,
    tavily: Optional[TavilyRetriever] = None,
    ddg: Optional[DuckDuckGoRetriever] = None,
    show_context: bool = False,
    confidence_threshold: float = 0.68
) -> Tuple[str, List[Dict]]:
    print(f"\n🔍 Query: {query}")
    if is_creative_or_grammar_query(query):
        cfg = {**generation_config, "temperature": 0.40}
        formatted = f"<start_of_turn>user\n{query}<end_of_turn>\n<start_of_turn>model\n"
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, **cfg)
        return extract_model_response(tokenizer.decode(out[0], skip_special_tokens=True)), []
    use_rag, initial, conf = should_use_rag(query, confidence_threshold)
    if not use_rag and initial.strip():
        print(f" ✅ Confident direct ({conf:.2f})")
        return initial, []
    print(f" 🔍 Retrieving (conf was {conf:.2f})...")
    articles = kb.search_and_fetch(query)
    context = ""
    sources = []
    if articles:
        kb.create_vector_store(articles)
        docs = kb.retrieve_relevant_context(query, k=3)
        for doc in docs:
            lang_note = " (Kiswahili)" if doc['language'] == 'sw' else ""
            context += f"[Wikipedia{lang_note} – {doc['title']}]\n{doc['content']}\n\n"
            sources.append({'title': doc['title'], 'url': doc['url'], 'language': doc['language']})
    if not context or (top_doc_is_generic(docs) and is_event_or_specific_query(query)):
        print(" 🌐 Web fallback...")
        res = tavily.search(query) if tavily else None
        if not res and ddg:
            res = ddg.search(query)
        if res:
            context = res['content']
            sources = [{'title': res['source'], 'url': res.get('url',''), 'language': 'en'}]
    if not context:
        return "Samahani, sikupata taarifa za kutosha kujibu swali hili.", []
    if show_context:
        print(f"\n📚 Context (truncated):\n{context[:600]}...\n")
    prompt = f"""Wewe ni msaidizi wa Kiswahili sahihi na wa kuaminika.
KANUNI (LAZIMA UFuate):
1. Answer in standard Kiswahili ONLY.
2. Use ONLY information provided in the CONTEXT. DO NOT CHANGE STATISTICS or create false ones.
3. FOR STATISTICS AND QUANTITIES: Use ACTUAL numbers from the DATA without changing even one.
4. STOP immediately after finishing the explanation.
5. DO NOT provide personal opinions unless the question clearly asks you to.
6. DO NOT REPEAT sentences or lists without reason. DO NOT LINK unrelated ideas.
7. Ikiwa jibu halipo kwenye CONTEXT, sema: "Samahani, maelezo yaliyotolewa hayatoshi."
9. FOR NUMBERS AND STATISTICS: Use the EXACT numbers from the PROVIDED INFORMATION without changing even a single digit. Do not write 'billion' or any large number unless it is written exactly that way in the PROVIDED INFORMATION.
10. Answer with only 50–120 words (approximately 3–5 sentences). STOP immediately after finishing the explanation.
11. Do NOT repeat sentences or lists without reason. Do NOT combine unrelated ideas.
TAARIFA:
{context}
SWALI: {query}
JIBU:"""
    formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    cfg = {**generation_config, "temperature": 0.15} # strong fact following
    with torch.no_grad():
        out = model.generate(**inputs, **cfg)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    cleaned = extract_model_response(full)
    return cleaned, sources
# =============================
# TEST RUN
# =============================
print("\n" + "="*70)
print("🚀 TESTING V5 — SWAHILI-GEMMA RAG SYSTEM")
print("="*70)
kb = WikipediaKnowledgeBase()
tavily = TavilyRetriever()
ddg = DuckDuckGoRetriever()
test_queries = [
    "Gavana wa Mombasa wa sasa ni nani?",
    "Idadi ya watu nchini Kenya inakadiriwa kuwa kiasi gani kufikia katikati ya mwaka 2025?",
    "Unadhani ni mji upi unaovutia zaidi kati ya Mombasa na Dar es Salaam kwa mtalii?",
    "hesabu ni somo nzuri?",
    "andika shairi fupi kuhusu bahari",
]
for i, q in enumerate(test_queries, 1):
    print(f"\n📝 {i}: {q}")
    print("-"*60)
    ans, srcs = generate_with_rag(q, kb, tavily, ddg, show_context=True)
    print(f"💬 Jibu: {ans}")
    if srcs:
        print("\n📚 Vyanzo:")
        for s in srcs:
            print(f" • {s['title']} → {s['url']}")
    print("-"*60)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


🚀 Using device: cuda

📥 Loading fine-tuned Swahili-Gemma model...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✅ Model loaded in 6.50 seconds

📥 Loading embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Embedding model loaded

⚙️ Generation Settings:
 • max_new_tokens: 220
 • do_sample: True
 • temperature: 0.3
 • top_p: 0.92
 • top_k: 50
 • repetition_penalty: 1.18

🚀 TESTING V5 — SWAHILI-GEMMA RAG SYSTEM

📝 1: Gavana wa Mombasa wa sasa ni nani?
------------------------------------------------------------

🔍 Query: Gavana wa Mombasa wa sasa ni nani?
 📊 Probe confidence rough: 0.95 | placeholder: False | short: False
 🔍 Retrieving (conf was 0.95)...
 🔍 Mapped → 'Abdulswamad Nassir'
 🔎 Searching: 'Abdulswamad Nassir'

📚 Context (truncated):
[Wikipedia (Kiswahili) – Abdullswamad Sherrif Nassir]
Abdullswamad Sheriff Nassir ni mwanasiasa wa Kenya na gavana wa kaunti ya Mombasa. Alichaguliwa kwa tiketi ya ODM chini ya azimio la Umoja, muungano wa Kenya kwenye uchaguzi mkuu wa 2022. Hapo awali alihudumu kama mbunge wa eneo Mvita kuanzia 2012 hadi 2022 .


== Marejeo ==

[Wikipedia – Abdullswamad Sherrif Nassir]
Abdullswamad Sheriff Nassir is a Kenyan politician. He is the governor of Momb